In [ ]:
import os

# Set working directory and GPU before importing JAX
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import vmap

from jaxpi.models import (
    create_lr_schedule,
    create_optimizer,
    create_arch,
    create_train_state,
)
from jaxpi.checkpointing import (
    create_checkpoint_manager,
    save_checkpoint,
    restore_checkpoint,
)

import models
from utils import get_dataset

jax.config.update("jax_default_matmul_precision", "highest")

In [ ]:
# Load config
from configs import baseline, fixed_pseudo_time, pseudo_time
config = baseline.get_config()

In [ ]:
# Get dataset
uv_ref, p_ref, temp_ref, t_ref, mesh, alpha1, alpha2, alpha3, alpha4, Ra, Pr, Ge = get_dataset(time_range=config.time_range)

# Initial condition of the first time window
u0 = uv_ref[0, :, 0]
v0 = uv_ref[0, :, 1]
temp0 = temp_ref[0, :]

# Get the time domain for each time window
num_time_steps = len(t_ref) // config.training.num_time_windows
t_star = t_ref[:num_time_steps]

# Define the time and space domain
dt = t_star[1] - t_star[0]
t0 = t_star[0]
t1 = t_star[-1] + 1.1 * dt

In [ ]:
# Initialize the model
lr = create_lr_schedule(config.optim)
tx = create_optimizer(config.optim, lr)
arch = create_arch(config.arch)
state = create_train_state(config, tx, arch)

# Initialize model
model = models.RayleighTaylor2D(config, lr, tx, arch, state, t_max=t1, alpha1=alpha1, alpha2=alpha2, alpha3=alpha3, alpha4=alpha4)

In [ ]:
# Create checkpoint manager for the first time window
window_idx = 0
ckpt_path = os.path.join(os.getcwd(), config.wandb.name, "ckpt")
ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path, suffix="time_window_{}".format(window_idx + 1))

# restore checkpoint
state = restore_checkpoint(ckpt_mngr, state)
print("Restore checkpoint from step: ", int(state.step))

In [ ]:
# Evaluate on test set
u_star = uv_ref[num_time_steps * window_idx: num_time_steps * (window_idx + 1), :, 0]
v_star = uv_ref[num_time_steps * window_idx: num_time_steps * (window_idx + 1), :, 1]
temp_star = temp_ref[num_time_steps * window_idx: num_time_steps * (window_idx + 1), :]

u_preds = []
v_preds = []
temp_preds = []
for t in t_star:
    print("Evaluating at time: ", t)
    u_pred, v_pred, _, T_pred = vmap( model.neural_net, in_axes=(None, None, 0, 0) )(state.params, t, mesh[:, 0], mesh[:, 1])

    u_preds.append(u_pred)
    v_preds.append(v_pred)
    temp_preds.append(T_pred)

u_preds = jnp.stack(u_preds, axis=0)
v_preds = jnp.stack(v_preds, axis=0)
temp_preds = jnp.stack(temp_preds, axis=0)

# Compute relative L2 error
u_error = jnp.linalg.norm(u_preds - u_star) / jnp.linalg.norm(u_star)
v_error = jnp.linalg.norm(v_preds - v_star) / jnp.linalg.norm(v_star)
temp_error = jnp.linalg.norm(temp_preds - temp_star) / jnp.linalg.norm(temp_star)
print("Relative L2 error for u: ", u_error)
print("Relative L2 error for v: ", v_error)
print("Relative L2 error for T: ", temp_error)

In [ ]:
# Plot results at the final time step
u_star = uv_ref[num_time_steps, :, 0].reshape(200, 100)
v_star = uv_ref[num_time_steps, :, 1].reshape(200, 100)
temp_star = temp_ref[num_time_steps].reshape(200, 100)

u_pred = u_preds[-1, :].reshape(200, 100)
v_pred = v_preds[-1, :].reshape(200, 100)
temp_pred = temp_preds[-1, :].reshape(200, 100)

x_star = mesh[:, 0].reshape(200, 100)
y_star = mesh[:, 1].reshape(200, 100)

plt.figure(figsize=(12, 5))
plt.subplot(1, 3, 1)
plt.title("Reference temperature")
plt.pcolor(x_star, y_star, temp_star, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 2)
plt.title("Predicted temperature")
plt.pcolor(x_star, y_star, temp_pred, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 3)
plt.title("Absolute error")
plt.pcolor(x_star, y_star, jnp.abs(temp_star - temp_pred), shading="auto", cmap="jet")
plt.colorbar()
plt.tight_layout()
plt.show()